<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%202/Aula_2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2.2 — Integrando tools com Agno

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 2 — Turbinando agentes com Tools

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 1.3** criamos o Treinador, sem tools.

Na **Aula 2.1** vimos que esse agente pode cometer erros relacionados a:

- **Desatualização**
- **Alucinação**  

Agora é hora de **plugar a primeira tool** no Treinador.

<br>

**O que vamos fazer:**

1. Recriar o Treinador e fazer uma **pergunta simples** — sem tool
2. Adicionar a primeira tool — **TavilyTools** para busca web
3. Refazer a mesma pergunta com tool e comparar
4. Observar o agente **decidindo não usar** a tool (pergunta conceitual)
5. Refinar instructions para guiar o uso


## Setup

Instalamos `agno`, `openai` e adicionamos `tavily-python` — a biblioteca cliente da Tavily, a API de busca web que vamos usar nesta aula.


In [ ]:
!pip install -U agno openai tavily-python

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### Cadastrando a API key do Tavily

- Resultados estruturados, otimizados pra LLMs consumirem
- Estável (sem rate limit aleatório como o de IPs compartilhados do Colab)
- Plano gratuito generoso (1.000 buscas/mês — sobra pro curso inteiro)

**Para usar nesta aula:**

Crie conta gratuita em [tavily.com](https://tavily.com)


---

## Passo 1 — Recriando o Treinador (sem tool, ainda)

Vamos partir do mesmo Treinador.


In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model= modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    markdown=True,
)

In [29]:
treinador.print_response(
    "Quantos gols o Vini Jr tem pela seleção?",
    stream=True,
)

Output()

---

## Passo 2 — Plugando a primeira tool




In [ ]:
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [30]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools

modelo_openai = OpenAIChat(id="gpt-5.4-mini")

treinador = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model= modelo_openai,
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",
    ],
    tools=[TavilyTools()],   # ← a única peça nova de capacidade
    markdown=True,
)

---

## Passo 3 — A mesma pergunta, agora com tool



In [31]:
treinador.print_response(
    "Quantos gols o Vini Jr tem pela seleção?",
    stream=True,
)

Output()

---

## Passo 4 — Observando o agente decidir *não* usar a tool



In [32]:
treinador.print_response(
    "Qual a vantagem tática de jogar com dois volantes contra um adversário que pressiona alto?",
    stream=True,
)

Output()

---

## Passo 5 — Refinando instructions para guiar o uso de tool

A decisão do agente sobre **quando usar tool** nem sempre é perfeita. Em testes você pode ver:

- O agente acionando tool para perguntas que ele já sabia responder (gasto desnecessário)
- O agente respondendo de cabeça quando deveria buscar (alucinação volta)

A correção é cirúrgica: **adicionar uma instruction** que oriente o uso, sem mexer nas outras.


In [33]:
treinador = Agent(
    name="Treinador",
    description="Assistente do ScoutAI FC sobre a Seleção Brasileira masculina de futebol.",
    model=OpenAIChat(id="gpt-5.4-mini"),
    instructions=[
        "Você é o Treinador, assistente do ScoutAI FC dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        # Nova instruction: guia o uso da tool
        "Use a busca web (Tavily) sempre que a pergunta envolver: "
        "eventos recentes (últimos jogos, convocações, lesões), "
        "estatísticas verificáveis (número de gols, recordes, classificações) "
        "ou forma atual de jogadores. "
        "Para perguntas conceituais (táticas, regras, princípios de jogo) "
        "ou históricas consolidadas (Copas antigas, técnicos do passado), "
        "responda direto sem busca.",
    ],
    tools=[TavilyTools()],
    markdown=True,
)

### Testando o refinamento

Duas perguntas, duas decisões esperadas:


In [34]:
# Pergunta com fato verificável recente → deve usar a tool
treinador.print_response(
    "Como o Vinícius Júnior tem se saído nos últimos jogos pelo Real Madrid?",
    stream=True,
)

Output()

In [35]:
# Pergunta tática conceitual → não deve usar a tool
treinador.print_response(
    "Em que situações um treinador prefere marcação por zona em vez de marcação individual?",
    stream=True,
)

Output()

---
### Estado atual do Treinador

```
Treinador (estado atual)
│
├── ✅ Identidade e tom (Aula 1.3)
├── ✅ Acesso a busca web (Aula 2.2 ← você está aqui)
├── ✅ Decide quando usar a tool (instruction nova)
│
├── ❌ Só tem uma fonte de dado          → Aula 2.3 (múltiplas tools)
├── ❌ Trabalha sozinho                  → Aula 3 (Olheiro + Analista)
├── ❌ Sem fluxo determinístico          → Aula 4 (Workflows)
└── ❌ Sem produção / monitoramento      → Aula 5 (Agent OS)
```

### O que vem agora

**Aula 2.3 — Tornando o agente mais inteligente com tools**

Uma tool resolve muita coisa, mas não tudo. Na próxima aula vamos **adicionar uma segunda tool** (Wikipedia, para fatos históricos estruturados) e refinar instructions para o Treinador **escolher entre fontes** — usando busca web para o que é recente e Wikipedia para o que é consolidado.
